# 本章的核心原則

> **任兩個待比較的策略之間必須僅存在單一變因**——此原則決定了整個實作架構。

- 形成期的**四層架構**使五個配對來源之間的差異被限縮在**分組層一處**
- 交易端利用「可在同一批配對上施行」的性質，使**動作空間得以被單獨消融**

第二章 2.3.5 指出動作空間未被當作實驗變因；
**本章 3.3 節把該觀察轉為可執行的設計。**

# 3.1 資料與時點處理

| 項目 | 設定 |
| :--- | :--- |
| 母體 | S&P 500 成分股（每期中位 411 檔） |
| 期間 | 2000-01-03 ~ 2025-12-31，**6,539 個交易日** |
| 可交易期間 | 扣首個形成窗後 **6,287 個交易日** |
| 價格 | 含息還原價（Tiingo） |
| 基本面 | FMP point-in-time 月頻；GICS 十一大產業 |

基本面覆蓋率隨時間變化甚大（2000–2008 **0%**、2015 62%、2024 72%），
全樣本兩欄同時有值僅 **31.9%**。

## 三處時點（point-in-time）控制

1. **成分股認定**——依 `index_memberships` 的納入／剔除日，
   每個形成期結束日僅保留**當時真實屬於 S&P 500** 的個股
   （843 檔、866 段成員期間，其中 **363 段已有剔除日**）
2. **基本面對齊**——一律取日期 ≤ 形成期結束日的最近一筆
3. **特徵計算窗**——僅用形成期窗內資料，交易期價格不參與任何估計

> 須區分：「**曾被剔出指數**」≠「**已下市**」。
> 存活者偏誤防範所需的是前者（已完整納入）。

## 滾動回測設計

| 參數 | 設定 |
| :--- | :--- |
| 形成期 | 252 個交易日 |
| 交易期 | **126 個交易日** |
| 滾動步長 | 21 個交易日 |
| 同時重疊期數 | **6**（= 126 / 21） |

任一時點有 6 個交易期同時運行 → **逐期報酬序列存在結構性自相關**
（決定 3.6 節的檢定設計）。

> **交易期長度亦決定了逐日自由持倉的決策次數**：
> 一期 126 個交易日 → 該動作空間每期最多 **126 次**決策，
> 而門檻選擇式每期僅 **1 次**。

## 交易成本假設與其雙重角色

單邊 **0.29%**（Do & Faff 2012 對美股的估計），進出場各扣一次
→ 一完整往返約 **0.58%**。

該估計樣本期為 1962–2009，套用於 2000–2025；
美股摩擦成本長期下降，故假設**偏保守**。

> **成本在本研究中有雙重角色**：既是績效的扣減項，
> 也是**動作空間消融的機制變數**——決策次數愈多、換手愈頻繁，
> 成本吞掉的比例愈高。故 H3 把「零成本重算」列為假說之一。

# 3.2 配對來源：四層架構與五個配對底

**本節所述各層不是被檢定的處理，而是配對的來源。**

```
特徵萃取 → 分組 → 統計篩選 → 群內排序
```

篩選置於排序**之前**，使組內每一組候選配對都受檢；
本研究**不設候選截斷**（否則檢定涵蓋範圍會與分組方法產生交互作用）。

此架構的作用：(1) 五個來源的差異被限縮在分組層一處；
(2) 交易端的對照可在任一來源上**原樣重複**。

## 特徵層：19 維（連續 7 維）

| 區塊 | 維度 | 內容 |
| :--- | :---: | :--- |
| 報酬主成分載荷 | 5 | 形成期日報酬 PCA 前 5 主成分 |
| 公司基本面 | 2 | 對數市值、盈餘殖利率 |
| GICS 產業 one-hot | 12 | 11 產業 ＋ 未知 |

> **關於加權**：$\sqrt{\lambda_j}$ 加權在實作上**不會**傳遞到分群距離——
> 三區塊各自 z 標準化後才拼接，而第 $j$ 欄的跨股標準差正比於 $\sqrt{\lambda_j}$，
> 該加權恰被除去。分群實際使用**等權**的 5 個主成分載荷。
> 等權化 = 對相關矩陣前 $k$ 維作球形化，是合理的表徵選擇，
> **惟其性質與「以因子重要性加權」相反**，據實陳述。

缺失值以**全域**中位數插補（不採產業中位數——那會成為隱性的產業資訊管道）。

## 分組層：五個配對來源

| 來源 | 排除比例 | 群數（中位） | 群大小（中位） |
| :--- | ---: | ---: | ---: |
| **不分組（NOGRP）** | **0%** | — | — |
| GICS | 15.1% | 13 | 22 |
| HDBSCAN | 26.4% | 14 | 14 |
| Agglomerative | 35.3% | 27 | 8 |
| K-means | 42.0% | 36 | 6 |

- K-means 的 $k$ **對齊同期 Agglomerative**，使比較聚焦於機制而非粒度
- HDBSCAN 的噪音點**無條件**併入 Unknown（噪音每期可達兩成，
  若僅依群大小判斷會被當成一個合法群）
- **逐期重估**（295 期各配適一次）：前視偏誤防範的必要條件，非受檢處理

## 分組結構的跨期穩定度（ARI）

| 分組方式 | 群數 | 相隔 1 期 | 相隔 6 期 |
| :--- | ---: | ---: | ---: |
| Agglomerative | 27 | 0.809 | 0.676 |
| HDBSCAN | 14 | 0.936 | 0.881 |
| K-means | 36 | 0.663 | 0.519 |
| **GICS（靜態）** | 13 | **1.000** | 1.000 |

> **HDBSCAN 的分布為雙峰，中位數不足以描述它**：
> 相鄰期 69.3% 的比較 ARI > 0.9，同時 **10.6% 的比較 ARI < 0.3**
> （最低 −0.006，與隨機無異），群數在 2–22 間跳動；
> 且該型態**不隨時間尺度累積**。

**範圍聲明**：本節僅刻畫穩定度，未檢定穩定與否何者有益。
配對層級週轉率**不可**用於推論分組層。

## 排序層：三種距離準則

皆建立在形成窗內**標準化的對數價格**之上。

- **SSD**：$\sum_t (P'_{i,t}-P'_{j,t})^2$
- **DTW**：Sakoe-Chiba 頻帶 $w=15$ 日，$O(Tw)$
- **SDP**：$z(SSD)+z(DTW)$

> **與 GGR 的差異**：該文最小化**累積總報酬指數**之差；本研究用對數價格的
> z 分數 → 對波動率不變，衡量的是**走勢形狀**而非**價格水準的共動**。
>
> **關於「主成分融合」**：$2\times2$ 相關矩陣的特徵向量恆為
> $\tfrac{1}{\sqrt2}(1,1)^\top$，**與 $\rho$ 無關** → 第一主成分恆等於等權和。
> 此處的 PCA **不提供任何資料驅動的加權自由度**，融合方向恆為 45 度。

**對沖比例兩個後端的估計式不同**：SSD 無截距、方向固定；
DTW／SDP 雙向 OLS 取 ADF $p$ 較小者。

## 篩選層：共整合檢定與一項臨界值的更正

Engle-Granger 兩步檢定，殘差作 ADF，$q=1$，$p<0.05$ 通過。

> **ADF 的 $p$ 值不可取自 Dickey-Fuller 分布。**
> DF 分布描述對一條**觀測到的**序列檢定單根；此處受檢的是**估計出來的**
> 共整合殘差，OLS 已先將其變異最小化，虛無分布位置明顯更負
> （Phillips & Ouliaris 1990）。改用 MacKinnon (2010) 響應曲面臨界值。
>
> **影響甚大**：沿用 DF 臨界值時通過率達 **77%**——
> 一個名目 5% 的檢定不可能拒絕 77% 的虛無假設；改用 EG 後降至約 **9.5%**。

**未採用兩項附加檢定**：OU 半衰期不具鑑別力（僅淘汰 0.2%）；
Hurst 前導實作**估計對象有誤**（R/S 定義在增量上，卻施於水準值）。

## 曝險的對齊（一項須先排除的干擾）

資金配置以「`top_n` × 重疊期數」為**固定分母**，與該期實際有幾組配對可用無關。
若各來源供應的配對數差異甚大，供應較少者將長期只部署較低比例的資本——
而檢定統計量為逐日報酬差，**與部署資本成正比**。

**實測顯示此干擾不成立**：每期選出配對數為
GICS 20.00、HDBSCAN 20.00、Agglomerative 20.00、K-means 19.79（上限 20），
填滿率皆 ≥ 98.9%。故維持固定分母的靜態配置。

# 3.3 動作空間的設計空間

**本節是本章的新增部分，也是本研究設計上的核心。**

| 類別 | 每期決策次數 | 動作集合 |
| :--- | :--- | :--- |
| 單一常數（不決策） | **0** | 固定 $(2.0, 0.0)$ |
| **每期選一組門檻** | **1** | SKIP ＋ 8 組 $(entry\_z, exit\_z)$，共 9 個 |
| **逐日決定持倉** | **≤ 126** | {做多價差, 做空價差, 空手} |

三者表達能力遞增：逐日決策可模擬任何固定門檻規則
→ 若僅以表達能力為準，它應**弱優於**門檻選擇。

> **本研究所要檢定的正是這個推論在日頻上是否成立。**

## 三代逐日自由持倉的實作

| 代 | 演算法 | 設計要點 |
| :--- | :--- | :--- |
| v1 | online DQN（LSTM 編碼狀態） | 逐步互動、經驗回放 |
| v2 | v1 的修復版 | 修掉訓練樣本共享與標籤對齊的缺陷 |
| v3 | FQI（批次擬合 Q） | 不做線上探索，批次反覆擬合 |

三代皆顯著劣於固定門檻基準，**且逐一修復訓練缺陷後劣勢依然存在**。

> **但開發過程中的觀察不構成論文證據**：
> (1) 當時三代與現行交易端並非跑在同一批配對上；
> (2) 該批回測的明細已於資料庫重建時遺失（附錄 B）。

## 消融的設計：唯一變因為動作空間

| 臂 | 動作空間 | 角色 |
| :--- | :--- | :--- |
| Z-Score | 單一常數 | 基準 |
| DL-THR（v4） | 每期選一組門檻 | 基準 |
| v1 / v2 / v3 | 逐日決定持倉 | 處理 |

| 控制項 | 設定 | 理由 |
| :--- | :--- | :--- |
| 配對來源 | `GICS-SDP` 單一來源 | 五臂共用同一批形成期配對 |
| 網格格點 | Top1／停損 0% | 無停損 → 強平不被截斷，觀察換手最乾淨 |
| 對沖口徑 | 一律 `dollar` | 封存模組未實作 signal 口徑 |
| 訓練預算 | 一律 **150 episodes** | **不得下修**——否則「訓練不足」成為無法排除的替代解釋 |

## 期間的選擇與一項結構性不對稱

**期間 2009-07 ~ 2018-12。** 逐日自由持倉的計算成本極高
（v1 實測逐期中位約 70 分鐘），全期 295 期不可行 → 取連續子期間。
在同成本的候選中選擇使**共同期落在樣本前後半分界兩側各半**的視窗。
**五臂共同期 108 期。**

> **只有 v1 需要形成期的價格切片**，故形成期起點早於價格索引起點的期
> （共 **12 期**，稱左緣期）只有 v1 算不了；其餘四臂各僅缺 1 期。
>
> 處置**寫定於執行前**：分析取**五臂共同期的交集**。
> 此不對稱在執行前即已揭露，非事後發現。

## 三個假說（執行前寫定）

> **H1（主要）** 逐日自由持倉的**換手率**顯著高於門檻選擇與固定門檻
>
> **H2（次要）** 逐日自由持倉的 Sharpe **不高於**門檻選擇
>
> **H3（機制）** 其績效劣勢**主要由換手成本造成**

**H3 的用意**：把「過度交易」與「因成本而虧損」拆開——
前者是動作空間的性質，後者是它的後果。若只量到後者，
這個消融就沒有說到動作空間。

> **明確不主張**：不主張門檻選擇式可交易、不主張任何一臂有正報酬。
> 此處量的是**相對的行為差異**，不是績效。

## 執行的核對：七項

> `run_trading` 回報的「成功」**不足以證明跑成功**——
> 本研究開發過程中曾兩度出現「回報成功但實際零有效模擬」（附錄 B）。

任一項不過即視為執行失敗，**不得就既有輸出做分析**。
核對以**落庫資料**為準（而非執行日誌——日誌是單次啟動尺度，
而本次跑跨多次續傳）：

1. 每期交易日列數完整　2. 五臂應算期集合一致　3. 共同期數下限
4. 逐期耗時（確認模型確實在訓練）　5. 對沖口徑
6. 命名無測試後綴　7. 落庫配對期中段無缺期

# 3.4 DL-THR：每期單次門檻選擇

**價差重建**（沿用形成期估得的參數，交易期不重新估計）：

$$s_t = P'_{A,t}-\beta P'_{B,t},\qquad z_t=\frac{s_t-\mu^{form}_{s}}{\sigma^{form}_{s}}$$

> **動作選單（9 個）**：SKIP ＋ 8 組
> $(entry\_z, exit\_z) \in \{1.5,2.0,2.5,3.0\}\times\{0.0,0.5\}$

三項結構性保證：

1. 選單**包含靜態基準** $(2.0,0.0)$ → 策略空間必然包含 Z-Score
2. 訓練樣本不足時**自動選用基準動作**
3. **SKIP** 使代理人可拒絕交易——固定規則不具備的選擇性

## 狀態空間與學習問題的性質

狀態為 **12 維形成期特徵**（偏離狀態、回歸品質、近期 regime、
配對性質、可交易性），全部可於交易期開始前計算。

> **全部 9 個動作的報酬都可精確反事實回算**
> （對該交易期的價格逐一模擬 9 組門檻）。
>
> 故此問題**並非部分回饋的 bandit，而是全資訊監督回歸**——不存在探索問題。
> MLP（12 → 64 → 9），MSE 損失，每期增量訓練 40 epoch。

**命名**：早期版本與程式碼稱其為「DRL」，依上述性質本文一律稱 **DL-THR**。

**前視防範**：walk-forward 增量訓練，期 $k$ 僅用**交易期已於期 $k$ 開始前結束**
的樣本；樣本 < 200 筆時自動選用基準動作。
網路未固定種子 → 三種 ML 來源各**五輪獨立重訓**。

# 3.5 RL-THR：部分回饋的受控對照

全資訊是否帶來優勢，是一項**待檢的預期而非既定事實**。

| | DL-THR | RL-THR |
| :--- | :--- | :--- |
| 觀測到的報酬 | 全部 9 個動作 | **僅實際選中的那一個** |
| 探索 | 不需要 | $\varepsilon$-greedy |
| 其餘設定 | — | **逐位元相同** |

> 兩臂的唯一差異是「是否觀測到未選動作的報酬」，
> 故其差額即為**反事實標籤的價值**。

# 3.6 評估與檢定設計

**抽樣單位**：以「參數網格」為觀測單位存在**偽重複**問題——
15 個配置共用同一份資料、期間與配對，有效樣本數接近**一條回測路徑**。
→ 改以**時間**為抽樣單位。

**主檢定**：逐日報酬差 ＋ 循環 block bootstrap（Künsch 1989；Politis & Romano 1992）

- 區塊長度 $L = 126$ 日（一個完整交易期），由 `FORWARD_DAYS` 直接決定
- 10,000 次重抽；平移分布施加 $H_0$；$p$ 值與信賴區間**同源**
- **未持倉日報酬記為 0**（該日確實沒有部位）

**為何不採 HAC**：全程並行計算作對照，兩法方向與量級一致；
選 bootstrap 的理由在**假設與研究者自由度**（HAC 依賴常態近似、須另指定落後階）。

## 消融的估計式（觀測單位是配對期）

| 項目 | 設定 |
| :--- | :--- |
| 主要結果變數 | 每期進場次數（**含反向**） |
| 檢定 | 逐期配對差分 ＋ 循環區塊拔靴，$L$ = **6 期** |
| 效果量 | **總進場數比值** |
| 達標 | 比值 ≥ 2.0 **且** BH 校正 $p<0.05$，且**須同時贏過兩個基準** |

> **為何效果量用總數比值而非逐期比值**：實測基準臂約 3% 的期零進場，
> 逐期比值會**除以零**；而剔除那些期恰好剔掉
> 「基準不交易、處理臂在交易」的期 → 系統性低估效果。

$L$ = 6 期使 108 期對應 **18 個獨立區塊**——本設計檢定力的實際上限。

## 報告口徑與預先登記

**一切績效主張與統計檢定皆以 15 個參數配置的等權組合為口徑。**
若改報「網格最佳配置」，該數字已內含「15 選 1」，
必須再以 DSR 之類的程序扣回——而其關鍵參數（試驗數 $N$）是**判斷而非事實**。
等權組合沒有東西可挑，選擇偏誤自源頭消失。

**一項例外**：第五章的 DSR 以網格最佳格計算（該節用途正是回答
「若改報最佳格會如何」）。實測**最佳格在成本維度上並非保守，而是偏樂觀**。

> **預先登記**：消融的假說、門檻、多重比較族、凍結超參與核對程序
> 皆於回測啟動前提交版本庫；偏離逐筆記於附錄 D。
>
> **此作法與本研究其餘部分不同**：主檢定的判準並非事前登記，
> 而是依文獻慣例於分析時採用。**兩者證據強度不同**，
> 第四、五章的陳述須依此區分。

# 3.7 參數敏感性分析

採 **OFAT**（每次只變動一個參數）。選取對象以「單獨即可能翻轉結論者」為準。

| 參數 | 基準 | 掃描值 |
| :--- | ---: | :--- |
| `hdbscan_min_cluster_size` | 5 | 3, 5, 10, 15 |
| `adf_pvalue_threshold` | 0.05 | 0.01, 0.05, 0.10 |
| `agg_threshold_percentile` | 75 | 50, 60, 75, 90, 95 |
| `entry_z` | 2.0 | 2.0, 2.1, 2.2, 2.3, 2.5, 3.0（**1.5 未執行**） |

> **`entry_z` 的實際覆蓋不均，須據實交代**：
> 2.0 共 53 個 METHOD、2.1–2.3 各 5 個、2.5／3.0 各 4 個，**1.5 從未執行**。
> 任何關於「門檻水準最適位置」的陳述**皆不得外推到 1.5**。

`entry_z` 另有專屬用途：分離「**動作空間的形狀**」與「**門檻水準**」
——第四章 4.4 以此為一項替代解釋。

## 本章小結

- 四層架構使五個配對來源的差異被限縮在分組層一處
- 交易端利用「同一批配對」的性質，使**動作空間得以被單獨消融**
- 以時間取代參數網格為抽樣單位；單一主檢定；全網格等權口徑
- 消融另以**配對期**為觀測單位、總進場數比值為效果量，並**完整預先登記**

> **實作與宣告的一致性**：本章所述方法，其實作曾出現多處與宣告不符
> （ADF 誤用 DF 臨界值、Hurst 測錯對象、HDBSCAN 噪音未排除、
> 產業中位數插補、候選截斷、交易統計計數器、對沖口徑、子視窗槽位洩漏），
> **均已修正，第四章全部數字皆產自修正後的管線**。詳見附錄 B。